# 31 — Does beam search help single-concept steering?

Beam search and the λ fluency blend were introduced and evaluated entirely
on multi-concept composition (E1/E1b/E3b), where beam was the largest
single improvement. Every single-concept measurement in the repo — the
paper's original method, the showcase `single` rows (24/29), the pool
validation (27) — uses the original greedy top-k loop. So two things are
unmeasured: whether beam's advantage transfers to the single-concept
setting, and whether the greedy singles used as the reference ceiling in
every multi-vs-single comparison understate that ceiling.

**Design: 2×2 factorial** on the 27 incumbent concepts, k=4, steps=24, the
10 showcase prompts, AWQ Llama (all matching notebook 24):

| search \ λ | 0.0 | 0.4 |
|---|---|---|
| greedy | **reused from notebook 24** (no angle metrics there) | run here |
| beam | run here | run here |

Fresh cells log expression + angle metrics (`angles=True`); the reused
greedy-λ0 cell has expression only, so angle comparisons cover the three
fresh cells while expression covers all four.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
CELLS = [("greedy", 0.4), ("beam", 0.0), ("beam", 0.4)]  # (search, lambda)

MODEL_ID = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"
MODEL_KEY = "llama-3.1-8b"


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/31_single_beam.jsonl")
LOG_PATH.unlink(missing_ok=True)

# greedy lambda=0 cell: notebook 24's single rows, reused (identical design)
GREEDY0 = [
    r
    for r in (
        json.loads(line)
        for line in Path("resources/24_showcase.jsonl").read_text().strip().split("\n")
    )
    if r["config"]["method"] == "single"
]
CONCEPTS = sorted({r["config"]["concept"] for r in GREEDY0})
print(f"{len(CONCEPTS)} concepts, {len(GREEDY0)} reused greedy-lambda0 records")

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(device_map="cuda:0", id=MODEL_ID)
prompts = [chat(u, a) for u, a in QUESTIONS]

harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)
concept = {
    name: fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS) for name in CONCEPTS
}
print(f"{len(concept)} concepts built")

for name in CONCEPTS:
    c = concept[name]
    for search, lam in CELLS:
        torch.cuda.empty_cache()
        if search == "greedy":
            # the canonical wrapper (generate_with_topk_guide) has no fluency
            # dial by design (golden-pinned paper function); the unified greedy
            # loop with a single guide is the sanctioned path to it
            texts, probe = fur.generate_with_topk_multi_guide(
                prompts, guides=[c], k=K, steps=STEPS, fluency_weight=lam
            )
        else:
            texts, probe = fur.generate_with_topk_beam_guide(
                prompts,
                concepts=[c],
                k=K,
                steps=STEPS,
                fluency_weight=lam,
                batch_size=4,
            )
        harness.evaluate(
            prompts,
            texts,
            [name],
            config={
                "model": MODEL_KEY,
                "method": "single",
                "search": search,
                "lam": lam,
                "concept": name,
            },
            probe=probe,
            baseline_texts=baseline_texts,
            concepts={name: c},
            angles=True,
        )
    print(f"{name}: done")

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
assert len(parsed) == len(CONCEPTS) * len(CELLS) * len(QUESTIONS), len(parsed)

rows = []
for r in parsed:
    c = r["config"]
    name = c["concept"]
    rows.append(
        {
            "search": c["search"],
            "lam": c["lam"],
            "concept": name,
            "delta": r["expression"][name]["delta"],
            "d_cos": r["angles"][name]["delta"]["token_cos"],
            "ppl_ratio": r["ppl_ratio"],
        }
    )
for r in GREEDY0:
    name = r["config"]["concept"]
    rows.append(
        {
            "search": "greedy",
            "lam": 0.0,
            "concept": name,
            "delta": r["expression"][name]["delta"],
            "d_cos": None,  # notebook 24 predates metric v3
            "ppl_ratio": r["ppl_ratio"],
        }
    )
results = pd.DataFrame(rows)
print(
    f"gate: {len(results)} records across {results.groupby(['search', 'lam']).size().to_dict()}"
)

summary = (
    results.groupby(["search", "lam"])
    .agg(
        delta=("delta", "mean"),
        positive=("delta", lambda s: (s > 0).mean()),
        d_cos=("d_cos", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n=("delta", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/31_single_beam_summary.csv", index=False)
summary.round(4)

In [ ]:
labels = [("greedy", 0.0), ("greedy", 0.4), ("beam", 0.0), ("beam", 0.4)]
names = [f"{s}\nlam={lam}" for s, lam in labels]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (metric, label) in zip(
    axes[:2],
    (("delta", "mean single-concept delta"), ("ppl_ratio", "mean ppl ratio")),
):
    vals = [
        summary[(summary["search"] == s) & (summary["lam"] == lam)][metric].iloc[0]
        for s, lam in labels
    ]
    ax.bar(
        range(len(labels)),
        vals,
        color=["tab:gray", "tab:cyan", "tab:blue", "tab:purple"],
    )
    ax.set_xticks(range(len(labels)), names, fontsize=8)
    ax.set_ylabel(label)
    ax.axhline(0, color="gray", lw=0.8)

# paired per-concept comparison: greedy lam=0 vs beam lam=0
per_concept = (
    results.groupby(["concept", "search", "lam"])["delta"].mean().unstack([1, 2])
)
g0, b0 = per_concept[("greedy", 0.0)], per_concept[("beam", 0.0)]
axes[2].scatter(g0, b0, s=18)
lim = [min(g0.min(), b0.min()) - 0.1, max(g0.max(), b0.max()) + 0.1]
axes[2].plot(lim, lim, color="gray", lw=0.8, ls="--")
axes[2].set_xlabel("greedy lam=0 delta (per concept)")
axes[2].set_ylabel("beam lam=0 delta (per concept)")

plt.suptitle("Single-concept steering: greedy vs beam search (27 concepts)")
plt.tight_layout()
plt.savefig("resources/31_single_beam.png", dpi=150)
plt.show()

wins = (b0 > g0).sum()
print(f"beam lam=0 beats greedy lam=0 on {wins}/{len(per_concept)} concepts")
print(f"mean improvement: {float((b0 - g0).mean()):+.3f}")

In [ ]:
def answer(text: str) -> str:
    marker = "assistant<|end_header_id|>"
    return text.split(marker)[-1].strip() if marker in text else text


best = (b0 - g0).idxmax()
print(f"largest beam gain: {best} ({float(b0[best] - g0[best]):+.2f})\n")
for search, lam in (("beam", 0.0), ("beam", 0.4)):
    for r in parsed:
        c = r["config"]
        if (
            c["concept"] == best
            and c["search"] == search
            and c["lam"] == lam
            and "short story" in r["prompt"]
        ):
            print(f"[{search} lam={lam}] delta={r['expression'][best]['delta']:+.2f}")
            print("   ", answer(r["text"])[:180], "\n")
            break
for r in GREEDY0:
    if r["config"]["concept"] == best and "short story" in r["prompt"]:
        print(f"[greedy lam=0.0] delta={r['expression'][best]['delta']:+.2f}")
        print("   ", answer(r["text"])[:180])
        break